# ConvTran: Multiscale Front-End Experiments

We test whether a multiscale temporal front-end changes ConvTran's accuracy, training time or memory use. Both variants use tAPE and eRPE, the released ConvTran configuration, and differ only in the first temporal convolution. For each run we record test accuracy, the time per training epoch and the peak CUDA memory during training and validation.

The `upstream/ConvTran` submodule is used at its pinned commit and is not modified. The multiscale layer is defined in `scripts/multiscale_frontend.py`.

| Variant | First temporal convolution | Filters |
|---|---|---|
| original | One Conv2d with kernel (1, 8), as in the upstream code | 64 |
| multiscale | Three parallel Conv2d branches with kernels (1, 3), (1, 7) and (1, 15) | 16 + 32 + 16 = 64 |

In the multiscale variant each branch uses stride 1, padding that keeps the temporal length, and a bias, as the upstream convolution does. The branch outputs are concatenated along the filter axis and pass through the same BatchNorm2d and GELU as in the original. The second convolution, which mixes the input channels, and the rest of the model are unchanged.

Both convolutions have 512 weights and 64 biases. The total number of trainable parameters is therefore the same for both variants.

Both variants share the upstream loader and its cached preprocessing, the fixed stratified validation split (`random_state=1234`), the RAdam optimizer and learning rate, and the evaluation on the official test split of the checkpoint with the lowest validation loss. For a given seed, all layers except the first convolution start from the same weights, and the random state used for shuffling and dropout is the same in both variants. Synthetic checks are in `tests/test_multiscale_frontend.py`.

## Configuration

The notebook has two modes. The pilot trains both variants on BasicMotions for two epochs with seed 1234 to check the pipeline and the measurements. The comparison trains both variants on Handwriting, PEMS-SF and FaceDetection for 100 epochs with the five baseline seeds.

| Setting | Pilot | Comparison |
|---|---|---|
| Datasets | BasicMotions | Handwriting, PEMS-SF, FaceDetection |
| Seeds | 1234 | 1234, 2345, 3456, 4567, 5678 |
| Epochs | 2 | 100 |
| Batch size | 16 | 16 |
| Embedding dimension | 16 | 16 |
| Attention heads | 8 | 8 |
| Runs | 2 | 30 |

`MODE` selects the mode and is set to the pilot. Training starts only when `RUN_TRAINING` is `True`; otherwise Run All reads the saved results. This cell also checks the runner and the submodule commit.

In [8]:
from pathlib import Path
import shlex
import subprocess
import sys

import pandas as pd

MODE = "comparison"  # "pilot" or "comparison"
RUN_TRAINING = True  # Set to True, then Run All, to train the selected mode.

VARIANTS = ["original", "multiscale"]
EXPECTED_SEEDS = [1234, 2345, 3456, 4567, 5678]
BATCH_SIZE = 16
EMB_SIZE = 16
NUM_HEADS = 8
MODES = {
    "pilot": {"datasets": ["BasicMotions"], "seeds": [1234], "epochs": 2},
    "comparison": {
        "datasets": ["Handwriting", "PEMS-SF", "FaceDetection"],
        "seeds": EXPECTED_SEEDS,
        "epochs": 100,
    },
}
assert MODE in MODES, f"MODE must be one of {sorted(MODES)}"
DATASETS = MODES[MODE]["datasets"]
SEEDS = MODES[MODE]["seeds"]
EPOCHS = MODES[MODE]["epochs"]

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)
assert ROOT is not None, "Open this notebook from the project repository."

RUNNER = ROOT / "scripts" / "run_multiscale_experiment.py"
UPSTREAM = ROOT / "upstream" / "ConvTran"
DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
assert RUNNER.is_file(), f"Missing experiment runner: {RUNNER}"

pinned = subprocess.check_output(
    ["git", "-C", str(ROOT), "ls-tree", "HEAD", "upstream/ConvTran"], text=True
).split()[2]
checked_out = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
).strip()
upstream_changes = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "status", "--porcelain"], text=True
).strip()
assert checked_out == pinned, f"Submodule at {checked_out}, pinned {pinned}"
assert not upstream_changes, "The upstream submodule has local changes."

sys.path.insert(0, str(ROOT / "scripts"))
from run_baseline import read_result
from run_positional_experiment import read_metrics
from run_multiscale_experiment import FIELDS, RESULTS_PATH, run_directory
from multiscale_frontend import FRONTENDS, frontend_tag

def run_dir(dataset, variant, seed):
    return run_directory(
        variant, dataset, seed, EPOCHS, BATCH_SIZE, EMB_SIZE, NUM_HEADS
    )

def log_complete(log):
    return read_result(log) is not None and read_metrics(log) is not None

def runner_args(dataset, variant, seed):
    return [
        "--variant", variant,
        "--dataset", dataset,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--emb-size", str(EMB_SIZE),
        "--num-heads", str(NUM_HEADS),
    ]

def retry_command(dataset, variant, seed):
    python = Path(sys.executable)
    if python.is_relative_to(ROOT):
        python = python.relative_to(ROOT)
    return shlex.join([
        str(python), str(RUNNER.relative_to(ROOT)),
        *runner_args(dataset, variant, seed), "--retry",
    ])

def as_available(table):
    return table.astype(object).where(table.notna(), "not available")

print("Repository:", ROOT)
print("Python:", sys.executable)
print("Upstream commit (pinned, clean):", pinned)
print("Results CSV:", RESULTS_PATH.relative_to(ROOT))
print("Front-ends:", ", ".join(f"{v} = {frontend_tag(v)}" for v in VARIANTS))
print(f"Mode: {MODE} ({EPOCHS} epochs, seeds {SEEDS})")
print("RUN_TRAINING:", RUN_TRAINING)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python
Upstream commit (pinned, clean): 148afb6ca549915b7e78b05e2ec3b4ba6e052341
Results CSV: results/multiscale_frontend_runs.csv
Front-ends: original = k8x64, multiscale = k3x16-k7x32-k15x16
Mode: comparison (100 epochs, seeds [1234, 2345, 3456, 4567, 5678])
RUN_TRAINING: True


## Front-end check

This check builds both variants on the CPU with a synthetic input of 6 channels. It does not use data or train. For each temporal length, it compares the output of the first stage (convolution, BatchNorm2d and GELU), the output of the second convolution and the model output. It also counts the trainable parameters of the first convolution and of the whole model.

The lengths include odd and even values and a single time step, since the original kernel of length 8 pads asymmetrically.

In [9]:
import torch
import multiscale_frontend

CHECK_CHANNELS, CHECK_CLASSES, CHECK_BATCH = 6, 4, 2

def check_config(length):
    return {
        "Net_Type": ["C-T"], "emb_size": EMB_SIZE, "num_heads": NUM_HEADS,
        "dim_ff": 256, "dropout": 0.01,
        "Data_shape": (CHECK_BATCH, CHECK_CHANNELS, length),
        "num_labels": CHECK_CLASSES,
    }

shape_rows = []
for length in (1, 8, 29, 100):
    x = torch.randn(CHECK_BATCH, CHECK_CHANNELS, length)
    for variant in VARIANTS:
        torch.manual_seed(0)
        model = multiscale_frontend.model_factory(check_config(length), variant).eval()
        with torch.no_grad():
            stage = model.embed_layer(x.unsqueeze(1))
            mixed = model.embed_layer2(stage)
            output = model(x)
        assert stage.shape == (CHECK_BATCH, 4 * EMB_SIZE, CHECK_CHANNELS, length)
        assert mixed.shape == (CHECK_BATCH, EMB_SIZE, 1, length)
        assert output.shape == (CHECK_BATCH, CHECK_CLASSES)
        shape_rows.append({
            "length": length, "variant": variant,
            "first_stage": tuple(stage.shape),
            "second_conv": tuple(mixed.shape),
            "model_output": tuple(output.shape),
        })
display(pd.DataFrame(shape_rows))

def count(module, name):
    return sum(p.numel() for n, p in module.named_parameters() if n.endswith(name))

parameter_rows = []
for variant in VARIANTS:
    torch.manual_seed(0)
    model = multiscale_frontend.model_factory(check_config(100), variant)
    first = model.embed_layer[0]
    parameter_rows.append({
        "variant": variant,
        "front_end": frontend_tag(variant),
        "first_conv_weights": count(first, "weight"),
        "first_conv_biases": count(first, "bias"),
        "first_conv_trainable": multiscale_frontend.trainable_parameters(first),
        "model_trainable": multiscale_frontend.trainable_parameters(model),
    })
parameters = pd.DataFrame(parameter_rows)
assert (parameters["first_conv_weights"] == 512).all()
assert (parameters["first_conv_biases"] == 64).all()
assert parameters["model_trainable"].nunique() == 1
display(parameters)
print("Both variants keep the temporal length and have the same number of trainable parameters.")

,length,variant,first_stage,second_conv,model_output
0,1,original,"(2, 64, 6, 1)","(2, 16, 1, 1)","(2, 4)"
1,1,multiscale,"(2, 64, 6, 1)","(2, 16, 1, 1)","(2, 4)"
2,8,original,"(2, 64, 6, 8)","(2, 16, 1, 8)","(2, 4)"
3,8,multiscale,"(2, 64, 6, 8)","(2, 16, 1, 8)","(2, 4)"
4,29,original,"(2, 64, 6, 29)","(2, 16, 1, 29)","(2, 4)"
5,29,multiscale,"(2, 64, 6, 29)","(2, 16, 1, 29)","(2, 4)"
6,100,original,"(2, 64, 6, 100)","(2, 16, 1, 100)","(2, 4)"
7,100,multiscale,"(2, 64, 6, 100)","(2, 16, 1, 100)","(2, 4)"


,variant,front_end,first_conv_weights,first_conv_biases,first_conv_trainable,model_trainable
0,original,k8x64,512,64,576,17884
1,multiscale,k3x16-k7x32-k15x16,512,64,576,17884


Both variants keep the temporal length and have the same number of trainable parameters.


## Run plan

The plan contains every combination of dataset, variant and seed in the selected mode. The cell builds the plans for both modes, checks that they contain 2 and 30 runs, and checks that the dataset files exist.

In [10]:
PLANS = {
    mode: [
        (dataset, variant, seed)
        for dataset in settings["datasets"]
        for seed in settings["seeds"]
        for variant in VARIANTS
    ]
    for mode, settings in MODES.items()
}
assert len(PLANS["pilot"]) == 2, len(PLANS["pilot"])
assert len(PLANS["comparison"]) == 30, len(PLANS["comparison"])
for mode, mode_plan in PLANS.items():
    assert len(set(mode_plan)) == len(mode_plan)
    settings = MODES[mode]
    print(
        f"{mode}: {len(mode_plan)} runs ({len(settings['datasets'])} datasets x "
        f"{len(VARIANTS)} variants x {len(settings['seeds'])} seeds, "
        f"{settings['epochs']} epochs)"
    )

for dataset in sorted({d for settings in MODES.values() for d in settings["datasets"]}):
    for split in ("TRAIN", "TEST"):
        path = DATA_ROOT / dataset / f"{dataset}_{split}.ts"
        assert path.is_file(), f"Missing dataset file: {path}"

plan = PLANS[MODE]
print(f"\nSelected: {MODE}, {len(plan)} runs")

pilot: 2 runs (1 datasets x 2 variants x 1 seeds, 2 epochs)
comparison: 30 runs (3 datasets x 2 variants x 5 seeds, 100 epochs)

Selected: comparison, 30 runs


## Training

When training is enabled, runs are executed one at a time, and each prints its progress, dataset, variant, seed and log path. Each run calls `scripts/run_multiscale_experiment.py`, which starts the upstream `main.py` in a separate process with its own log.

Logs, checkpoints and configurations are saved under `outputs/experiments/multiscale_frontend/<variant>/<dataset>/seed<seed>_ep<epochs>_bs<batch>_emb<emb>_h<heads>_fe-<front-end>/`. The front-end tag lists kernel and filters per branch, for example `k3x16-k7x32-k15x16`. Completed runs are added to `results/multiscale_frontend_runs.csv`, apart from the results of earlier notebooks. A run is reused only if its log for the same configuration is complete, so finished runs are not trained again. Pilot and comparison runs have different epochs and therefore different directories.

If a run fails, the cell prints the error, the log path and the retry command, then moves on. The incomplete log is kept, and later attempts stop at that run until it is retried explicitly. The `--retry` command renames the old log to `run.failed-<UTC time>.log` before training.

In [11]:
attempts = []
if not RUN_TRAINING:
    print("Training disabled; set RUN_TRAINING = True in the configuration cell.")

for index, (dataset, variant, seed) in enumerate(plan if RUN_TRAINING else [], 1):
    log = run_dir(dataset, variant, seed) / "run.log"
    print(
        f"\n[{index}/{len(plan)}] {dataset}, {variant}, seed {seed}\n"
        f"  log: {log.relative_to(ROOT)}",
        flush=True,
    )
    process = subprocess.run(
        [sys.executable, str(RUNNER), *runner_args(dataset, variant, seed)],
        cwd=ROOT, capture_output=True, text=True, check=False,
    )
    if process.stdout.strip():
        print(process.stdout.strip())
    if process.returncode:
        print(f"  FAILED with exit code {process.returncode}")
        print(process.stderr.strip())
        print(f"  Log: {log}")
        print(f"  Retry: {retry_command(dataset, variant, seed)}")
    attempts.append({
        "dataset": dataset,
        "variant": variant,
        "seed": seed,
        "exit_code": process.returncode,
    })

if attempts:
    attempts = pd.DataFrame(attempts)
    print(f"\nAttempted {len(attempts)} runs; {int((attempts['exit_code'] != 0).sum())} failed.")
    display(attempts[attempts["exit_code"] != 0])


[1/30] Handwriting, original, seed 1234
  log: outputs/experiments/multiscale_frontend/original/handwriting/seed1234_ep100_bs16_emb16_h8_fe-k8x64/run.log


Training original on Handwriting. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/multiscale_frontend/original/handwriting/seed1234_ep100_bs16_emb16_h8_fe-k8x64/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/multiscale_frontend_runs.csv

[2/30] Handwriting, multiscale, seed 1234
  log: outputs/experiments/multiscale_frontend/multiscale/handwriting/seed1234_ep100_bs16_emb16_h8_fe-k3x16-k7x32-k15x16/run.log
Training multiscale on Handwriting. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/multiscale_frontend/multiscale/handwriting/seed1234_ep100_bs16_emb16_h8_fe-k3x16-k7x32-k15x16/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/multiscale_frontend_runs.csv

[3/30] Handwriting, original, seed 2345
  log: outputs/experiments/multiscale_frontend/original/handwriting/seed23

,dataset,variant,seed,exit_code


## Results by run

Results are read from `results/multiscale_frontend_runs.csv` and the run directories, so this cell also works with training disabled. Only rows matching the selected front-ends, epochs, batch size, embedding dimension, number of heads, datasets and seeds are used.

A run is complete when the CSV has a row for its exact configuration and its log contains both the final test summary and the measurement record. A run is incomplete when its log exists but is not complete; it needs the `--retry` command. All other runs are pending.

Test accuracy and test loss come from the upstream evaluation of the checkpoint with the lowest validation loss on the official test split. Epoch times and peak memory are measured as in `04_positional_encoding_experiments.ipynb`. The first epoch is reported separately because it includes one-time costs. The later-epoch time is the mean over epochs 2 to the last. Peak memory is `torch.cuda.max_memory_allocated` over training and validation, and it is not available for CPU runs.

In [12]:
results = (
    pd.read_csv(RESULTS_PATH) if RESULTS_PATH.is_file()
    else pd.DataFrame(columns=FIELDS)
)
config_fields = ["dataset", "variant", "seed"]
selected = results[
    (results["epochs"] == EPOCHS)
    & (results["batch_size"] == BATCH_SIZE)
    & (results["emb_size"] == EMB_SIZE)
    & (results["num_heads"] == NUM_HEADS)
    & results["dataset"].isin(DATASETS)
    & results["variant"].isin(VARIANTS)
    & results["seed"].isin(SEEDS)
]
selected = selected[selected["frontend"] == selected["variant"].map(frontend_tag)]
if selected.duplicated(config_fields).any():
    raise ValueError("Duplicate rows for one configuration in the results CSV.")
recorded = {
    (row.dataset, row.variant, int(row.seed)): row
    for row in selected.itertuples(index=False)
}

def epoch_times(value):
    if pd.isna(value):
        return []
    return [float(part) for part in str(value).split(";") if part]

rows = []
for dataset, variant, seed in plan:
    log = run_dir(dataset, variant, seed) / "run.log"
    record = recorded.get((dataset, variant, seed))
    row = {
        "dataset": dataset, "variant": variant, "seed": seed,
        "log": str(log.relative_to(ROOT)),
    }
    if record is not None and log_complete(log):
        times = epoch_times(record.train_epoch_times_s)
        row.update(
            status="complete",
            test_accuracy_pct=record.accuracy * 100,
            test_loss=record.test_loss,
            first_epoch_s=times[0] if times else None,
            later_epochs_mean_s=(
                sum(times[1:]) / len(times[1:]) if len(times) > 1 else None
            ),
            timed_epochs=len(times),
            peak_cuda_alloc_train_val_mib=record.peak_cuda_allocated_train_val_mib,
            device=record.device,
        )
    elif log.is_file() and not log_complete(log):
        row["status"] = "incomplete"
    else:
        row["status"] = "pending"
    rows.append(row)

RUN_COLUMNS = [
    "dataset", "variant", "seed", "status", "test_accuracy_pct", "test_loss",
    "first_epoch_s", "later_epochs_mean_s", "timed_epochs",
    "peak_cuda_alloc_train_val_mib", "device", "log",
]
NUMERIC_COLUMNS = [
    "test_accuracy_pct", "test_loss", "first_epoch_s",
    "later_epochs_mean_s", "peak_cuda_alloc_train_val_mib",
]
runs_table = pd.DataFrame(rows).reindex(columns=RUN_COLUMNS)
runs_table[NUMERIC_COLUMNS] = runs_table[NUMERIC_COLUMNS].apply(pd.to_numeric)
runs_table["timed_epochs"] = runs_table["timed_epochs"].astype("Int64")

counts = runs_table["status"].value_counts()
print(f"{MODE}: {len(runs_table)} planned")
for status in ("complete", "pending", "incomplete"):
    print(f"  {status}: {counts.get(status, 0)}")
with pd.option_context("display.max_rows", None):
    display(as_available(runs_table.round(4)))

incomplete = runs_table[runs_table["status"] == "incomplete"]
if not incomplete.empty:
    print("Incomplete logs are kept. Inspect each log, then retry from the repository root:")
    for run in incomplete.itertuples(index=False):
        print(" ", retry_command(run.dataset, run.variant, run.seed))

comparison: 30 planned
  complete: 30
  pending: 0
  incomplete: 0


,dataset,variant,seed,status,test_accuracy_pct,test_loss,first_epoch_s,later_epochs_mean_s,timed_epochs,peak_cuda_alloc_train_val_mib,device,log
0,Handwriting,original,1234,complete,21.7647,2.852,0.2077,0.0177,100,71.0161,cuda,outputs/experiments/multiscale_frontend/origin...
1,Handwriting,multiscale,1234,complete,22.5882,2.8029,0.2138,0.0196,100,71.02,cuda,outputs/experiments/multiscale_frontend/multis...
2,Handwriting,original,2345,complete,22.8235,2.7167,0.1913,0.0172,100,71.0161,cuda,outputs/experiments/multiscale_frontend/origin...
3,Handwriting,multiscale,2345,complete,24.7059,2.6877,0.2063,0.0187,100,71.02,cuda,outputs/experiments/multiscale_frontend/multis...
4,Handwriting,original,3456,complete,25.6471,2.6692,0.1971,0.0176,100,71.0161,cuda,outputs/experiments/multiscale_frontend/origin...
5,Handwriting,multiscale,3456,complete,29.1765,2.5039,0.2065,0.0192,100,71.02,cuda,outputs/experiments/multiscale_frontend/multis...
6,Handwriting,original,4567,complete,25.8824,2.6745,0.197,0.0176,100,71.0161,cuda,outputs/experiments/multiscale_frontend/origin...
7,Handwriting,multiscale,4567,complete,24.7059,2.7708,0.2099,0.0195,100,71.02,cuda,outputs/experiments/multiscale_frontend/multis...
8,Handwriting,original,5678,complete,23.5294,2.572,0.196,0.0173,100,71.0161,cuda,outputs/experiments/multiscale_frontend/origin...
9,Handwriting,multiscale,5678,complete,21.1765,2.7923,0.2083,0.0193,100,71.02,cuda,outputs/experiments/multiscale_frontend/multis...


## Summary by dataset and variant

Completed seeds are summarized for each dataset and variant. Accuracy is the mean in percent with the sample standard deviation in percentage points. The standard deviation is not available with a single seed.

Epoch times are first averaged over epochs 2 to the last within each seed, then across seeds. First-epoch time and peak memory are averaged across seeds.

Summaries with pending seeds are labelled as partial and should not be compared directly with complete ones.

In [13]:
summary_rows = []
for dataset in DATASETS:
    for variant in VARIANTS:
        done = runs_table[
            (runs_table["dataset"] == dataset)
            & (runs_table["variant"] == variant)
            & (runs_table["status"] == "complete")
        ]
        completed = len(done)
        summary_rows.append({
            "dataset": dataset,
            "variant": variant,
            "summary": "complete" if completed == len(SEEDS) else "partial",
            "seeds_completed": f"{completed}/{len(SEEDS)}",
            "mean_accuracy_pct": done["test_accuracy_pct"].mean(),
            "std_accuracy_pp": (
                done["test_accuracy_pct"].std(ddof=1) if completed > 1 else None
            ),
            "mean_first_epoch_s": done["first_epoch_s"].mean(),
            "mean_later_epochs_s": done["later_epochs_mean_s"].mean(),
            "mean_peak_cuda_alloc_train_val_mib": (
                done["peak_cuda_alloc_train_val_mib"].mean()
            ),
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_NUMERIC = summary.columns[4:]
summary[SUMMARY_NUMERIC] = summary[SUMMARY_NUMERIC].apply(pd.to_numeric)
display(as_available(summary.round(4)))

,dataset,variant,summary,seeds_completed,mean_accuracy_pct,std_accuracy_pp,mean_first_epoch_s,mean_later_epochs_s,mean_peak_cuda_alloc_train_val_mib
0,Handwriting,original,complete,5/5,23.9294,1.7912,0.1978,0.0175,71.0161
1,Handwriting,multiscale,complete,5/5,24.4706,3.027,0.2089,0.0192,71.02
2,PEMS-SF,original,complete,5/5,71.6763,1.7816,3.4037,3.2263,2750.8589
3,PEMS-SF,multiscale,complete,5/5,70.7514,3.9754,3.4769,3.2995,2750.4653
4,FaceDetection,original,complete,5/5,65.6981,1.5507,1.2476,1.0678,160.5176
5,FaceDetection,multiscale,complete,5/5,64.983,0.8994,1.3433,1.1391,159.9697


## Paired accuracy difference

For each dataset and seed where both variants are complete, we take the multiscale accuracy minus the original accuracy, in percentage points. A positive value favours the multiscale front-end. Both runs of a pair share the seed, the validation split and the initial weights of all layers except the first convolution.

The table reports the number of pairs, the mean difference and its sample standard deviation. Datasets with fewer than five pairs are labelled as partial. The per-seed differences are shown below the summary.

In [14]:
complete_runs = runs_table[runs_table["status"] == "complete"]
paired = (
    complete_runs.pivot_table(
        index=["dataset", "seed"], columns="variant", values="test_accuracy_pct"
    )
    .reindex(columns=VARIANTS)
    .dropna()
    .reindex(index=DATASETS, level="dataset")
    .rename_axis(columns=None)
    .reset_index()
)
paired["difference_pp"] = paired["multiscale"] - paired["original"]

difference_rows = []
for dataset in DATASETS:
    pairs = paired.loc[paired["dataset"] == dataset, "difference_pp"]
    difference_rows.append({
        "dataset": dataset,
        "summary": "complete" if len(pairs) == len(SEEDS) else "partial",
        "pairs": f"{len(pairs)}/{len(SEEDS)}",
        "mean_difference_pp": pairs.mean(),
        "std_difference_pp": pairs.std(ddof=1) if len(pairs) > 1 else None,
    })
differences = pd.DataFrame(difference_rows)
differences[["mean_difference_pp", "std_difference_pp"]] = (
    differences[["mean_difference_pp", "std_difference_pp"]].apply(pd.to_numeric)
)
print(f"Seeds with both variants complete: {len(paired)} of {len(DATASETS) * len(SEEDS)}")
display(as_available(differences.round(4)))
if not paired.empty:
    display(paired.round(4))

Seeds with both variants complete: 15 of 15


,dataset,summary,pairs,mean_difference_pp,std_difference_pp
0,Handwriting,complete,5/5,0.5412,2.3524
1,PEMS-SF,complete,5/5,-0.9249,4.1602
2,FaceDetection,complete,5/5,-0.7151,1.9218


,dataset,seed,original,multiscale,difference_pp
0,Handwriting,1234,21.7647,22.5882,0.8235
1,Handwriting,2345,22.8235,24.7059,1.8824
2,Handwriting,3456,25.6471,29.1765,3.5294
3,Handwriting,4567,25.8824,24.7059,-1.1765
4,Handwriting,5678,23.5294,21.1765,-2.3529
5,PEMS-SF,1234,71.6763,73.9884,2.3121
6,PEMS-SF,2345,72.8324,69.9422,-2.8902
7,PEMS-SF,3456,71.6763,75.7225,4.0462
8,PEMS-SF,4567,68.7861,67.0520,-1.7341
9,PEMS-SF,5678,73.4104,67.0520,-6.3584


## Limitations

The five seeds of each dataset share one validation split, and each timing comes from a single run on a desktop GPU. With five pairs per dataset, the mean difference is a rough estimate, and three datasets do not support a claim about the UEA archive as a whole.

The multiscale front-end keeps the number of filters and parameters, but not the receptive field. Its branches see 3, 7 and 15 time steps instead of 8, so any difference may come from the longer or shorter kernels rather than from combining scales. The comparison applies to this configuration and to an embedding dimension of 16.

The two-epoch pilot only checks that the pipeline and the measurements work. It is too short to compare accuracy.